# 08 RL Mini Project

## 목표
MountainCar에서 tabular RL과 naive DQN의 한계를 확인하고, replay buffer와 target network를 직접 넣어보며 DQN 안정화의 필요성을 이해한다.

## 수업 흐름
1. MountainCar 환경 관찰
2. Q-learning / SARSA with discretization
3. naive DQN의 실패와 불안정성 확인
4. replay buffer 추가
5. target network 추가
6. 결과 비교와 해석 정리


## MountainCar 한눈에 보기
- 상태(state): `[position, velocity]`
- 행동(action): `0=왼쪽`, `1=가속 안 함`, `2=오른쪽`
- 보상(reward): 매 step마다 `-1`
- 종료: 정상 도달 또는 `200 step`
- 핵심 난점: 왕복하며 속도를 누적해야 정상에 도달할 수 있다.

## 오늘 던질 질문
- 연속 상태를 Q-table에 바로 넣을 수 없는 이유는?
- discretization을 하면 어떤 정보가 사라지는가?
- naive DQN이 왜 불안정할까?
- replay buffer는 무엇을 완화하고, target network는 무엇을 고정할까?


In [ ]:
import random

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display

torch.set_num_threads(1)
plt.style.use("seaborn-v0_8")


In [ ]:
SEED = 7

TABULAR_CONFIG = {
    "n_position_bins": 20,
    "n_velocity_bins": 20,
    "episodes": 1500,
    "max_steps": 200,
    "alpha": 0.15,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "avg_window": 20,
    "log_every": 100,
}

NAIVE_DQN_CONFIG = {
    "episodes": 80,
    "max_steps": 200,
    "hidden_dim": 32,
    "lr": 1e-3,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "avg_window": 20,
    "log_every": 20,
    "loss_log_window": 200,
}

STABLE_DQN_CONFIG = {
    "episodes": 100,
    "max_steps": 200,
    "hidden_dim": 32,
    "lr": 5e-4,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "avg_window": 20,
    "log_every": 20,
    "loss_log_window": 50,
    "buffer_capacity": 5000,
    "batch_size": 32,
    "learn_start": 200,
    "train_frequency": 4,
    "target_sync_steps": 200,
}

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_env(seed=None):
    env = gym.make("MountainCar-v0")
    if seed is not None:
        env.action_space.seed(seed)
    return env

def linear_epsilon(episode_idx, total_episodes, start, end):
    progress = episode_idx / max(total_episodes - 1, 1)
    return max(end, start - (start - end) * progress)

def moving_average(values, window=20):
    values = np.asarray(values, dtype=np.float32)
    if len(values) == 0:
        return values
    if len(values) < window:
        return values.copy()
    weights = np.ones(window, dtype=np.float32) / window
    return np.convolve(values, weights, mode="valid")

def plot_reward_curves(reward_dict, title, avg_window=20):
    plt.figure(figsize=(10, 4))
    for label, rewards in reward_dict.items():
        rewards = list(rewards)
        plt.plot(rewards, alpha=0.25, label=f"{label} raw")
        smooth = moving_average(rewards, avg_window)
        start_idx = avg_window - 1 if len(rewards) >= avg_window else 0
        plt.plot(range(start_idx, start_idx + len(smooth)), smooth, linewidth=2, label=f"{label} MA({avg_window})")
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

def plot_loss_curves(loss_dict, title, avg_window=50):
    plt.figure(figsize=(10, 4))
    for label, losses in loss_dict.items():
        losses = list(losses)
        if not losses:
            continue
        smooth = moving_average(losses, avg_window)
        start_idx = avg_window - 1 if len(losses) >= avg_window else 0
        plt.plot(range(start_idx, start_idx + len(smooth)), smooth, linewidth=2, label=label)
    plt.title(title)
    plt.xlabel("Gradient step")
    plt.ylabel("Loss")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

def build_summary_row(name, state_repr, result, stability, note):
    recent_avg = float(np.mean(result["rewards"][-20:])) if result["rewards"] else np.nan
    return {
        "algorithm": name,
        "state_repr": state_repr,
        "last_reward": round(float(result["rewards"][-1]), 2),
        "recent_20_avg": round(recent_avg, 2),
        "goals_reached": result["goals_reached"],
        "first_success_episode": result["first_success_episode"] if result["first_success_episode"] is not None else "-",
        "stability": stability,
        "one_line_interpretation": note,
    }

def show_logs(label, result):
    print(f"[{label}]")
    display(pd.DataFrame(result["logs"]))

seed_everything(SEED)
print("Seed:", SEED)


In [ ]:
env = make_env(SEED)
obs_low = env.observation_space.low
obs_high = env.observation_space.high
print("Observation space:", env.observation_space)
print("Action space size:", env.action_space.n)
print("Observation low:", obs_low)
print("Observation high:", obs_high)
print("Goal position:", env.unwrapped.goal_position)
env.close()

tabular_bins = np.array([TABULAR_CONFIG["n_position_bins"], TABULAR_CONFIG["n_velocity_bins"]])

def discretize_state(state, obs_low=obs_low, obs_high=obs_high, bins=tabular_bins):
    state = np.array(state, dtype=np.float32)
    ratios = (state - obs_low) / (obs_high - obs_low)
    clipped = np.clip(ratios, 0.0, 0.999999)
    indices = (clipped * bins).astype(int)
    return tuple(indices)

def select_action_from_q(q_table, discrete_state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()
    return int(np.argmax(q_table[discrete_state]))

sample_env = make_env(SEED)
sample_state, _ = sample_env.reset(seed=SEED)
print("Sample state:", sample_state)
print("Discretized state:", discretize_state(sample_state))
sample_env.close()


In [ ]:
def train_tabular(method="q_learning", config=TABULAR_CONFIG, seed=SEED):
    assert method in {"q_learning", "sarsa"}
    seed_everything(seed)
    env = make_env(seed)
    bins = np.array([config["n_position_bins"], config["n_velocity_bins"]])
    q_table = np.zeros((bins[0], bins[1], env.action_space.n), dtype=np.float32)
    rewards, logs = [], []
    goals_reached = 0
    first_success_episode = None

    for episode in range(1, config["episodes"] + 1):
        state, _ = env.reset(seed=seed + episode)
        discrete_state = discretize_state(state, obs_low, obs_high, bins)
        epsilon = linear_epsilon(episode - 1, config["episodes"], config["epsilon_start"], config["epsilon_end"])
        if method == "sarsa":
            action = select_action_from_q(q_table, discrete_state, epsilon, env)
        episode_reward = 0.0

        for _ in range(config["max_steps"]):
            if method == "q_learning":
                action = select_action_from_q(q_table, discrete_state, epsilon, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            next_discrete_state = discretize_state(next_state, obs_low, obs_high, bins)
            episode_reward += reward

            if method == "q_learning":
                best_next_q = np.max(q_table[next_discrete_state])
                td_target = reward if done else reward + config["gamma"] * best_next_q
                q_table[discrete_state][action] += config["alpha"] * (td_target - q_table[discrete_state][action])
            else:
                if done:
                    td_target = reward
                    q_table[discrete_state][action] += config["alpha"] * (td_target - q_table[discrete_state][action])
                else:
                    next_action = select_action_from_q(q_table, next_discrete_state, epsilon, env)
                    td_target = reward + config["gamma"] * q_table[next_discrete_state][next_action]
                    q_table[discrete_state][action] += config["alpha"] * (td_target - q_table[discrete_state][action])
                    action = next_action

            discrete_state = next_discrete_state
            if done:
                if next_state[0] >= env.unwrapped.goal_position:
                    goals_reached += 1
                    if first_success_episode is None:
                        first_success_episode = episode
                break

        rewards.append(episode_reward)
        if episode % config["log_every"] == 0:
            recent_avg = float(np.mean(rewards[-config["avg_window"] :]))
            logs.append({"episode": episode, "reward": round(float(episode_reward), 2), "recent_avg": round(recent_avg, 2), "epsilon": round(float(epsilon), 3), "goals": goals_reached})
            print(f"[{method}] episode={episode:4d}, reward={episode_reward:6.1f}, recent_avg={recent_avg:7.2f}, epsilon={epsilon:5.3f}, goals={goals_reached}")

    env.close()
    return {"q_table": q_table, "rewards": rewards, "logs": logs, "goals_reached": goals_reached, "first_success_episode": first_success_episode}


In [ ]:
q_learning_result = train_tabular("q_learning")
sarsa_result = train_tabular("sarsa")

plot_reward_curves({"Q-learning": q_learning_result["rewards"], "SARSA": sarsa_result["rewards"]}, "Tabular RL on MountainCar", avg_window=TABULAR_CONFIG["avg_window"])
show_logs("Q-learning", q_learning_result)
show_logs("SARSA", sarsa_result)


## Tabular 실험 메모
- Seed=`7`, 기본 설정에서 Q-learning은 episode `1500` 기준 recent_avg `-186.40`, goals `32` 정도를 보였다.
- 같은 설정에서 SARSA는 episode `1500` 기준 recent_avg `-180.60`, goals `130` 정도를 보였다.
- discretization 덕분에 연속 상태를 표로 바꿀 수 있지만, 상태 정보가 거칠어진다.
- 이번 단계의 핵심은 solve가 아니라 “연속 상태를 tabular 방식으로 다루기 위해 어떤 손실이 생기는가”를 보는 것이다.


## 2. naive DQN
이제 raw continuous state를 그대로 넣는 DQN으로 넘어간다.
단, 이번 버전은 replay buffer와 target network를 일부러 빼 둔 naive한 형태다.


In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)

def state_to_tensor(state):
    return torch.tensor(state, dtype=torch.float32).unsqueeze(0)

def select_action_dqn(state, epsilon, q_network, env):
    if random.random() < epsilon:
        return env.action_space.sample()
    state_tensor = state_to_tensor(state)
    with torch.no_grad():
        q_values = q_network(state_tensor)
    return int(q_values.argmax(dim=1).item())


In [ ]:
def train_naive_dqn(config=NAIVE_DQN_CONFIG, seed=SEED):
    seed_everything(seed)
    env = make_env(seed)
    online_net = QNetwork(env.observation_space.shape[0], env.action_space.n, config["hidden_dim"])
    optimizer = optim.Adam(online_net.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    rewards, losses, logs = [], [], []
    goals_reached = 0
    first_success_episode = None

    for episode in range(1, config["episodes"] + 1):
        state, _ = env.reset(seed=seed + episode)
        epsilon = linear_epsilon(episode - 1, config["episodes"], config["epsilon_start"], config["epsilon_end"])
        episode_reward = 0.0

        for _ in range(config["max_steps"]):
            action = select_action_dqn(state, epsilon, online_net, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            state_tensor = state_to_tensor(state)
            next_state_tensor = state_to_tensor(next_state)
            q_values = online_net(state_tensor)
            predicted_q = q_values[0, action]

            with torch.no_grad():
                next_q_values = online_net(next_state_tensor)
                max_next_q = next_q_values.max(dim=1).values.item()
                td_target = reward if done else reward + config["gamma"] * max_next_q

            target_tensor = torch.tensor(td_target, dtype=torch.float32)
            loss = loss_fn(predicted_q, target_tensor)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))
            episode_reward += reward
            state = next_state
            if done:
                if next_state[0] >= env.unwrapped.goal_position:
                    goals_reached += 1
                    if first_success_episode is None:
                        first_success_episode = episode
                break

        rewards.append(episode_reward)
        if episode % config["log_every"] == 0:
            recent_avg = float(np.mean(rewards[-config["avg_window"] :]))
            recent_losses = losses[-config["loss_log_window"] :]
            loss_mean = float(np.mean(recent_losses)) if recent_losses else np.nan
            loss_std = float(np.std(recent_losses)) if recent_losses else np.nan
            logs.append({"episode": episode, "reward": round(float(episode_reward), 2), "recent_avg": round(recent_avg, 2), "epsilon": round(float(epsilon), 3), "loss_mean": round(loss_mean, 4), "loss_std": round(loss_std, 4), "goals": goals_reached})
            print(f"[naive_dqn] episode={episode:3d}, reward={episode_reward:6.1f}, recent_avg={recent_avg:7.2f}, loss_mean={loss_mean:8.4f}, loss_std={loss_std:8.4f}, goals={goals_reached}")

    env.close()
    return {"online_net": online_net, "rewards": rewards, "losses": losses, "logs": logs, "goals_reached": goals_reached, "first_success_episode": first_success_episode}


In [ ]:
naive_dqn_result = train_naive_dqn()

plot_reward_curves({"Naive DQN": naive_dqn_result["rewards"]}, "Naive DQN Reward Curve", avg_window=NAIVE_DQN_CONFIG["avg_window"])
plot_loss_curves({"Naive DQN": naive_dqn_result["losses"]}, "Naive DQN Loss Curve", avg_window=50)
show_logs("Naive DQN", naive_dqn_result)


## naive DQN 예시 실패 로그
- Seed=`7`, 기본 설정에서 episode `80` 기준 recent_avg는 `-198.75`, goals는 `1` 수준이었다.
- 같은 설정에서 loss_mean은 `17.6109`, loss_std는 `246.2528`까지 커졌다.
- 예측과 target 계산을 같은 네트워크가 맡고, 직전 transition 하나로만 업데이트하기 때문에 불안정성이 크게 나타난다.


## 3. replay buffer와 target network 추가
이번 단계에서는 `replay only`와 `replay + target`을 비교해 본다.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        transition = (np.array(state, dtype=np.float32), int(action), float(reward), np.array(next_state, dtype=np.float32), float(done))
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


In [ ]:
def train_stabilized_dqn(use_target_network=True, config=STABLE_DQN_CONFIG, seed=SEED):
    seed_everything(seed)
    env = make_env(seed)
    online_net = QNetwork(env.observation_space.shape[0], env.action_space.n, config["hidden_dim"])
    target_net = QNetwork(env.observation_space.shape[0], env.action_space.n, config["hidden_dim"])
    target_net.load_state_dict(online_net.state_dict())
    optimizer = optim.Adam(online_net.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    replay_buffer = ReplayBuffer(config["buffer_capacity"])
    rewards, losses, logs = [], [], []
    goals_reached = 0
    first_success_episode = None
    total_steps = 0
    variant_name = "replay+target" if use_target_network else "replay_only"

    for episode in range(1, config["episodes"] + 1):
        state, _ = env.reset(seed=seed + episode)
        epsilon = linear_epsilon(episode - 1, config["episodes"], config["epsilon_start"], config["epsilon_end"])
        episode_reward = 0.0

        for _ in range(config["max_steps"]):
            action = select_action_dqn(state, epsilon, online_net, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            replay_buffer.push(state, action, reward, next_state, done)
            total_steps += 1
            episode_reward += reward

            enough_samples = len(replay_buffer) >= config["batch_size"]
            can_learn = total_steps >= config["learn_start"]
            learn_now = total_steps % config["train_frequency"] == 0

            if enough_samples and can_learn and learn_now:
                states, actions, rewards_batch, next_states, dones = replay_buffer.sample(config["batch_size"])
                predicted_q = online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    next_source = target_net if use_target_network else online_net
                    max_next_q = next_source(next_states).max(dim=1).values
                    targets = rewards_batch + config["gamma"] * max_next_q * (1.0 - dones)
                loss = loss_fn(predicted_q, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(float(loss.item()))

            if use_target_network and total_steps % config["target_sync_steps"] == 0:
                target_net.load_state_dict(online_net.state_dict())

            state = next_state
            if done:
                if next_state[0] >= env.unwrapped.goal_position:
                    goals_reached += 1
                    if first_success_episode is None:
                        first_success_episode = episode
                break

        rewards.append(episode_reward)
        if episode % config["log_every"] == 0:
            recent_avg = float(np.mean(rewards[-config["avg_window"] :]))
            recent_losses = losses[-config["loss_log_window"] :]
            loss_mean = float(np.mean(recent_losses)) if recent_losses else np.nan
            loss_std = float(np.std(recent_losses)) if recent_losses else np.nan
            logs.append({"episode": episode, "reward": round(float(episode_reward), 2), "recent_avg": round(recent_avg, 2), "epsilon": round(float(epsilon), 3), "loss_mean": round(loss_mean, 4), "loss_std": round(loss_std, 4), "goals": goals_reached})
            print(f"[{variant_name}] episode={episode:3d}, reward={episode_reward:6.1f}, recent_avg={recent_avg:7.2f}, loss_mean={loss_mean:8.4f}, loss_std={loss_std:8.4f}, goals={goals_reached}")

    env.close()
    return {"online_net": online_net, "target_net": target_net, "rewards": rewards, "losses": losses, "logs": logs, "goals_reached": goals_reached, "first_success_episode": first_success_episode}


In [ ]:
replay_dqn_result = train_stabilized_dqn(use_target_network=False)
target_dqn_result = train_stabilized_dqn(use_target_network=True)

plot_reward_curves({"Naive DQN": naive_dqn_result["rewards"], "Replay only": replay_dqn_result["rewards"], "Replay + Target": target_dqn_result["rewards"]}, "DQN Variant Reward Comparison", avg_window=20)
plot_loss_curves({"Naive DQN": naive_dqn_result["losses"], "Replay only": replay_dqn_result["losses"], "Replay + Target": target_dqn_result["losses"]}, "DQN Variant Loss Comparison", avg_window=50)
show_logs("Replay only", replay_dqn_result)
show_logs("Replay + Target", target_dqn_result)


## 예시 비교 로그
- Seed=`7`, 기본 설정에서 replay only는 episode `100` 기준 loss_mean `14.8048`, loss_std `38.3323` 수준이었다.
- 같은 설정에서 replay + target은 episode `100` 기준 loss_mean `10.2902`, loss_std `26.5188` 수준이었다.
- replay buffer만으로는 target이 계속 움직이고, target network를 추가해야 TD target 진동이 더 줄어든다.
- 이번 주차 핵심은 solve가 아니라 실패 원인 추론과 안정화 장치의 역할 해석이다.


In [ ]:
comparison_rows = [
    build_summary_row("Q-learning", "20x20 discretized state", q_learning_result, "tabular, coarse aggregation", "discretization으로 적용 가능하지만 상태 정보가 거칠다."),
    build_summary_row("SARSA", "20x20 discretized state", sarsa_result, "tabular, on-policy", "Q-learning과 유사하지만 실제 정책 기반 업데이트라 더 보수적으로 보일 수 있다."),
    build_summary_row("Naive DQN", "raw continuous state", naive_dqn_result, "loss variance is very large", "연속 상태를 직접 다루지만 학습이 매우 불안정하다."),
    build_summary_row("DQN + Replay", "raw continuous state", replay_dqn_result, "correlation reduced, target still moving", "샘플 상관성은 줄지만 target이 흔들려 불안정성이 남아 있다."),
    build_summary_row("DQN + Replay + Target", "raw continuous state", target_dqn_result, "loss oscillation reduced", "target을 잠시 고정해 더 안정적인 업데이트를 만든다."),
]
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


## 학생 기록 템플릿
| Algorithm | State representation | Episode reward | Recent 20 avg | Goals reached | Stability | One-line interpretation |
|---|---|---:|---:|---:|---|---|
| Q-learning |  |  |  |  |  |  |
| SARSA |  |  |  |  |  |  |
| Naive DQN |  |  |  |  |  |  |
| DQN + Replay |  |  |  |  |  |  |
| DQN + Replay + Target |  |  |  |  |  |  |

## 자주 나는 오류와 힌트
- discretization index가 bin 범위를 넘는다면 `np.clip(..., 0.0, 0.999999)`을 다시 확인한다.
- Gymnasium은 `terminated`, `truncated`를 둘 다 확인해야 한다.
- DQN에서 `gather`를 쓸 때는 `actions.unsqueeze(1)`이 필요하다.
- replay buffer 길이가 `batch_size`보다 짧을 때 `sample()` 하면 에러가 난다.
- target network는 선언만 하면 끝이 아니라 주기적으로 복사해야 한다.

## 시간이 부족할 때
1. 튜닝보다 로그 해석을 우선한다.
2. 최소한 `naive DQN`과 `Replay + Target`은 비교한다.
3. 여유가 있으면 bin 수나 episode 수를 바꿔 추가 실험한다.
